## Tracking of a EURUSD dataset optimised over a parameter grid, using likelihoods

In [130]:
## Non-optimised params
optimise_over_T_timesteps= 2000 #If using binder, change to lower num of steps (200 confimred to work)
filter_over_T_timesteps= 2000
seed = 1 # Random seem for reproducibility
c=10

In [131]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater
from scipy.stats import norm
from scipy.special import logsumexp



In [ ]:
############################################################
# 0.  Imports (minimal – add any others you already have)
############################################################
import os, copy, numpy as np, pandas as pd
from datetime   import timedelta
from functools  import lru_cache
from scipy.stats      import lognorm
from scipy.optimize   import minimize
from scipy.special    import logsumexp

from stonesoup.types.state           import (GaussianState,
                                             MarginalisedParticleState)
from stonesoup.types.array           import StateVectors, CovarianceMatrices
from stonesoup.types.detection       import Detection
from stonesoup.types.track           import Track
from stonesoup.types.numeric         import Probability
from stonesoup.types.hypothesis      import SingleHypothesis

from stonesoup.resampler.particle    import SystematicResampler
from stonesoup.updater.kalman        import KalmanUpdater
from stonesoup.updater.particle      import MarginalisedParticleUpdater
from stonesoup.predictor.kalman      import KalmanPredictor
from stonesoup.predictor.particle    import MarginalisedParticlePredictor
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear      import RandomWalk
from stonesoup.models.transition.levy_linear import LevyRandomWalk
from stonesoup.models.driver         import AlphaStableNSMDriver
from stonesoup.models.transition.levy_linear import LevyNthDerivativeDecay, LevyLangevin
from stonesoup.models.transition.linear import NthDerivativeDecay, OrnsteinUhlenbeck
############################################################
# 1.  Load tick data  (EUR/CHF example)
############################################################
folder     = "TrackedDatasets"
file_name  = "EURCHF_Ticks_03.02.2025-03.02.2025.csv"
data       = pd.read_csv(os.path.join(folder, file_name))

tick_times = data['Local time']
mid_rates  = data['Mid']
ask_rates  = data['Ask']
bid_rates  = data['Bid']

filter_over_T_timesteps   = min(filter_over_T_timesteps, len(tick_times))
optimise_over_T_timesteps = min(optimise_over_T_timesteps,
                                filter_over_T_timesteps)

mid_rates = mid_rates.iloc[5000:5000+filter_over_T_timesteps].to_numpy()
bid_rates = bid_rates.iloc[5000:5000+filter_over_T_timesteps].to_numpy()
ask_rates = ask_rates.iloc[5000:5000+filter_over_T_timesteps].to_numpy()
fmt       = r"%d.%m.%Y %H:%M:%S.%f GMT%z"
timesteps = pd.to_datetime(tick_times.iloc[:filter_over_T_timesteps], format=fmt)
start_time = timesteps[0]


############################################################
# 2.  Empirical measurement noise (σₑ) and its prior
############################################################
diffs         = np.diff(mid_rates)
residuals     = diffs[np.abs(diffs) <= np.percentile(np.abs(diffs), 95)]
price_std     = np.std(residuals, ddof=1)

In [133]:
# variances used for numerical integration over σₑ²
sigma2=2e-5**2

In [134]:
# 3.  Build measurement models & detections  (keys = σₑ²)
############################################################
meas_model = LinearGaussian(
    ndim_state=1,           # price only
    mapping=(0,),
    noise_covar=np.atleast_2d([sigma2])
)


In [135]:
#  ✧  Observed price tracks & Detection list (ONE σₑ²)  ✧
#      – mid-price is used as the “measurement’’ –
# ──────────────────────────────────────────────────────────────
mid_measurements = Track()
observed_mid = Track()        # for plotting reference
observed_bid = Track()
observed_ask = Track()

measurements = []         # ← this replaces the old measurements_dict

# ensure bid/ask arrays are NumPy (makes indexing easy)
mid_arr = np.asarray(mid_rates)
bid_arr = np.asarray(bid_rates)
ask_arr = np.asarray(ask_rates)

for price_mid, price_bid, price_ask, ts in zip(mid_arr, bid_arr, ask_arr, timesteps):
    # ground-truth style tracks (useful for plotter)
    mid_measurements.append(Detection(
                state_vector=np.array([price_mid]),
                timestamp      = ts,
                measurement_model = meas_model))
    observed_ask.append(GroundTruthState([[price_ask]], timestamp=ts))
    observed_mid.append(GroundTruthState([[price_mid]], timestamp=ts))
    observed_bid.append(GroundTruthState([[price_bid]], timestamp=ts))


In [202]:
# 4.  Prior states (shared between models)
############################################################
number_particles = 2000
# prior_mean       = np.array([mid_rates[0],0])
prior_mean       = np.array([mid_rates[0]])

prior_sig_pos   = (bid_rates[0]-ask_rates[0])**2
# prior_covar      = np.diag([prior_sig_pos**2,
#                             1e-2*prior_sig_pos**2])
covars = np.full((1, 1, number_particles), prior_sig_pos)
states=multivariate_normal.rvs(prior_mean,prior_sig_pos,size=number_particles).reshape(number_particles, 1)

lp_prior = Track(MarginalisedParticleState(
    state_vector = StateVectors(states.T),
    covariance   = CovarianceMatrices(covars),
    weight       = np.full(number_particles, 1/number_particles),
    timestamp    = start_time - timedelta(milliseconds=1)
))

gp_prior = Track(GaussianState(state_vector=prior_mean,
                               covar=np.atleast_2d(prior_sig_pos),
                               timestamp=start_time - timedelta(milliseconds=1)))

In [203]:
# 6.  Cached builders (transition+updaters)
############################################################
@lru_cache(maxsize=None)
def build_gp_objects(sigw2,theta):
    # model   = OrnsteinUhlenbeck(noise_diff_coeff=sigw2,
    #                             damping_coeff=theta)
    model  = RandomWalk(noise_diff_coeff=sigw2)
    
    pred    = KalmanPredictor(model)
    updater =KalmanUpdater(meas_model)
    return pred, updater

@lru_cache(maxsize=None)
def build_lp_objects(sigw2,theta,alpha):
    driver = AlphaStableNSMDriver(mu_W=0, sigma_W2=sigw2,
                                  c=10.0, alpha=alpha,
                                  noise_case=NoiseCase(2))
    # model  = LevyLangevin(driver=driver, noise_diff_coeff=sigw2,
    #                       damping_coeff=theta)
    model  = LevyRandomWalk(driver=driver, noise_diff_coeff=sigw2)
    pred   = MarginalisedParticlePredictor(model)
    resamp = SystematicResampler()
    updater = MarginalisedParticleUpdater(meas_model,
                                                resamp)
    return pred, updater


############################################################
# 7.  Log-likelihood (p(y|θ) with σₑ² integrated out)
############################################################
def filter_loglike(params, model='gp',track=None):
    if model == 'gp':
        sigw2,theta = params
        predictor, updater = build_gp_objects(sigw2,theta)
        base_track = gp_prior
    else:
        sigw2, theta, alpha = params
        predictor, updater = build_lp_objects(sigw2,theta, alpha)
        base_track = lp_prior

    logl    =  - np.log(optimise_over_T_timesteps) # logprior_sigma2

    trk = [base_track[0]]  # fresh copy each σ₂
    
    for meas in mid_measurements[:optimise_over_T_timesteps]:
        pred  = predictor.predict(trk[-1], timestamp=meas.timestamp)
        hypo  = SingleHypothesis(pred, meas)
        post  = updater.update(hypo)
        trk.append(post)

        mp      = hypo.measurement_prediction
        
        # 1-D measurement → flatten to (N,) where N = #particles (N=1 for GP)
        means   = np.asarray(mp.state_vector).ravel()           # (N,)
        covattr = 'covariance' if hasattr(mp, 'covariance') else 'covar'
        vars_   = np.asarray(getattr(mp, covattr)).reshape(-1)  # (N,)

        y       = meas.state_vector.item()

        if means.size == 1:          # ---- Kalman / GP case --------------
            mu   = means[0]
            var  = vars_[0]
            logl += -0.5*((y-mu)**2/var + np.log(2*np.pi*var)) 
            
        else:                        # ---- Particle (MPF) case ------------
            ll_arr = -0.5*((y - means)**2/vars_ + np.log(2*np.pi*vars_))
            logl += logsumexp(ll_arr) - np.log(means.size) 
        
    return logl

In [265]:
sigma_w2_gp=1e-4**2 
theta_gp= 2.5

sigma_w2_lp= 9e-6**2
theta_lp= 0.12
alpha= 1.3
mu=0

sig2=1e-10

In [ ]:

# # 4. Plot histogram + Gaussian & Lévy overlays
# # --------------------------------------------
# from matplotlib import pyplot as plt
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from scipy.stats import norm, levy_stable
# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go
# R = sigma2
# Q = 1e-8  # a tiny process‐noise variance so smoother can denoise

# N = len(mid_rates)
# # Allocate arrays
# x_pred = np.zeros(N)
# P_pred = np.zeros(N)
# x_filt = np.zeros(N)
# P_filt = np.zeros(N)

# # Initialize filter
# x_filt[0] = mid_rates[0]
# P_filt[0] = 1.0  # large initial uncertainty

# # (a) Forward Kalman filter
# for k in range(1, N):
#     # Predict
#     x_pred[k] = x_filt[k-1]
#     P_pred[k] = P_filt[k-1] + Q

#     # Update
#     K = P_pred[k] / (P_pred[k] + R)
#     x_filt[k] = x_pred[k] + K * (mid_rates[k] - x_pred[k])
#     P_filt[k] = (1 - K) * P_pred[k]

# # (b) Backward RTS smoother
# x_smooth = np.zeros(N)
# P_smooth = np.zeros(N)
# x_smooth[-1] = x_filt[-1]
# P_smooth[-1] = P_filt[-1]

# for k in range(N - 2, -1, -1):
#     C = P_filt[k] / (P_pred[k + 1])
#     x_smooth[k] = x_filt[k] + C * (x_smooth[k + 1] - x_pred[k + 1])
#     P_smooth[k] = P_filt[k] + C * (P_smooth[k + 1] - P_pred[k + 1]) * C

# # --------------------------------------------
# # 4. Compute smoothed returns: Δx_smooth[k] = x_smooth[k] - x_smooth[k-1]
# # --------------------------------------------
# # 1. Compute smoothed returns
# smoothed_returns = np.diff(data['Mid'])

# # 2. Define overlay parameters
# returns = diffs  # if you intend to use raw-return range for x-axis
# scale_levy = np.sqrt(sigma_w2_lp)

# # 3. Build a fine grid for PDF overlays (using smoothed_returns range)
# x_min, x_max = smoothed_returns.min(), smoothed_returns.max()
# x_vals = np.linspace(x_min, x_max, 1000)

# # Gaussian PDF (µ=0, variance = sigma_w2_gp)
# pdf_gaussian = norm.pdf(x_vals, loc=0, scale=np.sqrt(sigma_w2_gp))
# pdf_gaussian2 = norm.pdf(x_vals, loc=0, scale=np.sqrt(sig2))


# # Lévy‐stable PDF (symmetric, β=0)
# pdf_levy = levy_stable.pdf(x_vals, alpha, 0, loc=0, scale=scale_levy)

# # 4. Create Plotly figure
# fig = go.Figure()

# # (a) Histogram of smoothed returns as density
# fig.add_trace(
#     go.Histogram(
#         x=smoothed_returns,
#         nbinsx=300,
#         histnorm='probability density',
#         name='Smoothed Returns',
#         marker_color='blue',
#         opacity=0.6,
#         hovertemplate='Return: %{x:.6f}<br>Density: %{y:.6f}<extra></extra>'
#     )
# )
# # (b) Gaussian PDF overlay
# fig.add_trace(
#     go.Scatter(
#         x=x_vals,
#         y=pdf_gaussian2,
#         mode='lines',
#         line=dict(color='grey', dash='dash', width=2),
#         name=f'Gaussian (σ²={sig2:.1e})'
#     )
# )

# # (b) Gaussian PDF overlay
# fig.add_trace(
#     go.Scatter(
#         x=x_vals,
#         y=pdf_gaussian,
#         mode='lines',
#         line=dict(color='grey', dash='dash', width=2),
#         name=f'Gaussian (σ²={sigma_w2_gp:.1e})'
#     )
# )

# # (c) Lévy‐stable PDF overlay
# fig.add_trace(
#     go.Scatter(
#         x=x_vals,
#         y=pdf_levy,
#         mode='lines',
#         line=dict(color=colors[0], width=1),
#         name=f'Lévy‐stable (α={alpha}, c≈{scale_levy:.1e})'
#     )
# )

# # 5. Update layout to match your style
# fig.update_layout(
#     plot_bgcolor="white",
#     width=800,height=800,
#     xaxis=dict(
#         showgrid=False,
#         title=dict(text="Time-adjusted 'Returns'", font=dict(size=10))
#     ),
#     yaxis=dict(
#         showgrid=False,
#         title=dict(text="Density", font=dict(size=10))
#     ),
#     title=dict(
#         text="Histogram of Smoothed Returns with Gaussian and Lévy‐stable Overlays",
#         font=dict(size=22),
#         x=0.5
#     ),
#     legend=dict(
#         font=dict(size=12),
#         bordercolor="Black",
#         borderwidth=2,
#         orientation='h',
#         y=-0.1
#     )
# )

# # 6. Show the interactive figure
# fig.show()

In [241]:
# 8.  Objectives for SciPy (negative log-posterior)
############################################################
def obj_gp(x):
    sigw2 = np.exp(x[0])
    theta = x[1]             
    print(np.sqrt(sigw2),theta)
    return -filter_loglike((sigw2,theta), model='gp')

def obj_lp(x):
    sigw2 = np.exp(x[0])
    theta = x[1]               
    alpha = x[2]
    if np.abs(alpha-1.0)<0.01:
        if np.random.random()<0.5:
            alpha=x[2]=0.95
        else:
            alpha=x[2]=1.05
    print(np.sqrt(sigw2),theta,alpha)
    return -filter_loglike((sigw2,theta, alpha), model='lp')


def build_init_simplex(model):
    if model=='gp':
        x0=np.array([sigma_w2_gp,theta_gp])
    elif model=='lp':
        x0=np.array([sigma_w2_lp,theta_lp,alpha])

    # build a “large” initial simplex by offsetting each coordinate by ±Δ:
    n = len(x0)

    # SciPy expects an (n+1)×n array where each row is a vertex.
    initial_simplex = np.zeros((n+1, n))
    initial_simplex[0] = np.array(x0)

    for i in range(n):
        if i==0:
            delta=1.4
        else:
            delta=0.1
        # move ±Δ along the i-th axis (you can also use different Δ_i per axis)
        vertex = x0.copy()
        vertex[i] += delta
        initial_simplex[i+1] = vertex

############################################################
# 9.  Nelder–Mead optimisation
############################################################
# print("\n=== Gaussian RW optimisation ===")
# res_gp = minimize(obj_gp,
#                   x0=[np.log(sigma_w2_gp),theta_gp],
#                   method='Nelder-Mead',
#                   options={'maxiter': 1000, 'disp': True,
#                            "initial_simplex": build_init_simplex('lp')},
#                   bounds=[(None,None),(0,None)])

# sigma_gp_opt, theta_gp_opt= (np.exp(res_gp.x[0]/2),
#                              res_gp.x[1])
# print("Best σ_w (GP):", sigma_gp_opt, "best theta:", theta_gp_opt)


# print("\n=== Lévy RW optimisation ===")
# res_lp = minimize(obj_lp,
#                   x0=[np.log(sigma_w2_lp),theta_lp,alpha],
#                   method='Nelder-Mead',
#                   options={'maxiter': 1000, 'disp': True,
#                            "initial_simplex": build_init_simplex('lp')},
#                            bounds=[(None,None),(1,1),(0.05,1.95)])

# (sigma_lp_opt, theta_lp_opt,alpha_opt)= (np.exp(res_lp.x[0]/2),
#                                           res_lp.x[1], 
#                                           res_lp.x[2])

# print("Best σ_w (LP):", sigma_lp_opt,"best theta:",theta_lp_opt,"best alpha:" ,alpha_opt)

In [267]:
from stonesoup.plotter import Plotterly, Dimension

from stonesoup.plotter import Plotterly, Dimension
from stonesoup.types.groundtruth import GroundTruthPath


driver_x = AlphaStableNSMDriver(mu_W=mu, sigma_W2=sigma_w2_lp, seed=None, c=c, alpha=alpha, noise_case=NoiseCase.GAUSSIAN_APPROX)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta_lp)
langevin_x = LevyRandomWalk(driver=driver_x)

transition_model = langevin_x

truth = GroundTruthPath([GroundTruthState([mid_arr[0]], timestamp=timesteps[0])])
# The state of the target can be represented as 2D Cartesian coordinates, $\left[x, \dot x, y, \dot y\right]^{\top}$, modelling both its position and velocity. A simple truth path is created with a sampling rate of 1 Hz.

for t,timestep in enumerate(timesteps):
    truth.append(GroundTruthState(
        transition_model.function(truth[-1], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[t]))
        
# vplotter = Plotterly(autosize=False, width=1500, height=800,
#                     dimension=Dimension.ONE, axis_labels=["Price"])
# # observed mid / bid / ask
# vplotter.plot_ground_truths(truth, [1], truths_label="generated")

# # ---- aesthetics --------------------------------------------------
# vplotter.fig.update_layout(
#     plot_bgcolor="white",
#     xaxis=dict(showgrid=True, gridcolor="lightgray",
#                title=dict(text="Time", font=dict(size=20))),
#     yaxis=dict(showgrid=True, gridcolor="lightgray",
#                title=dict(text="Price", font=dict(size=20))),
#     legend=dict(font=dict(size=12),
#                 bordercolor="Black", borderwidth=2,
#                 orientation='h')
# )
# vplotter.fig.show()
plotter = Plotterly(autosize=False, width=1500, height=800,
                    dimension=Dimension.ONE, axis_labels=["Price"])
# observed mid / bid / ask
plotter.plot_ground_truths(observed_mid, [0], truths_label="Mid")
# plotter.plot_ground_truths(truth, [0], truths_label="generated")

# plotter.plot_ground_truths(observed_bid, [0], truths_label="Bid")
# plotter.plot_ground_truths(observed_ask, [0], truths_label="Ask")

# ---- aesthetics --------------------------------------------------
plotter.fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray",
               title=dict(text="Time", font=dict(size=20))),
    yaxis=dict(showgrid=True, gridcolor="lightgray",
               title=dict(text="Rate", font=dict(size=20))),
    legend=dict(font=dict(size=12),
                bordercolor="Black", borderwidth=2,
                orientation='h')
)

In [268]:
# 4b.  Likelihood:
# ------------------------------------------------------------------
gp_l=filter_loglike((sigma_w2_gp,theta_gp),'gp')
print(f'gp likelihood= {gp_l}')
lp_l=filter_loglike((sigma_w2_lp,theta_lp,alpha),'lp')
print(f'lp likelihood= {lp_l}')

gp likelihood= 17665.688692689135
lp likelihood= 18412.659514533392


In [269]:
# inference
from stonesoup.smoother.particle import MarginalisedKalmanSmoother
gp_predictor , gp_updater = build_gp_objects(sigma_w2_gp,theta_gp)    # <- SINGLE object, not a dict
lp_predictor, lp_updater= build_lp_objects(sigma_w2_lp,theta_lp,alpha)

LP_track = Track(copy.deepcopy(lp_prior[0]))
GP_track = Track(copy.deepcopy(gp_prior[0]))

# ------------------------------------------------------------------
# 4.  Single-pass filtering for the *whole* data set
# ------------------------------------------------------------------
for i, meas in enumerate(mid_measurements):
    # ---- Lévy (MPF) update ---------------------------------------
    lp_pred = lp_predictor.predict(LP_track[-1], timestamp=meas.timestamp)
    lp_hypo = SingleHypothesis(lp_pred, meas)
    lp_post = lp_updater.update(lp_hypo)
    LP_track.append(lp_post)

    # ---- Gaussian (Kalman) update --------------------------------
    gp_pred = gp_predictor.predict(GP_track[-1], timestamp=meas.timestamp)
    gp_hypo = SingleHypothesis(gp_pred, meas)
    gp_post = gp_updater.update(gp_hypo)
    GP_track.append(gp_post)
    if i/100==i//100:
        print(f'measurement {i}/{filter_over_T_timesteps}')

colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']

measurement 0/2000
measurement 100/2000
measurement 200/2000
measurement 300/2000
measurement 400/2000
measurement 500/2000
measurement 600/2000
measurement 700/2000
measurement 800/2000
measurement 900/2000
measurement 1000/2000
measurement 1100/2000
measurement 1200/2000
measurement 1300/2000
measurement 1400/2000
measurement 1500/2000
measurement 1600/2000
measurement 1700/2000
measurement 1800/2000
measurement 1900/2000


In [270]:
# 5.  Plotting
# ------------------------------------------------------------------
from pathlib import Path

folder_path = r"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"
file_path   = Path(folder_path) / "eurchfpriceplot.html"
file_path.parent.mkdir(parents=True, exist_ok=True)

# filtered tracks
plotter.plot_tracks(LP_track, [0], mode="lines",
                    uncertainty=True, particle=False,
                    track_label="Lévy filter", line=dict(width=1,color=colors[0]))
plotter.plot_tracks(GP_track, [0], mode="lines",
                    uncertainty=True, particle=False,
                    track_label="Gauss filter", line=dict(width=1,dash='dot',color='grey'))

# plotter.fig.write_html(str(file_path))   # ⇐ uncomment to save
plotter.fig.show()

In [271]:
# RTS_track=MarginalisedKalmanSmoother().smooth(LP_track[1:])
# plotter.plot_tracks(RTS_track, [0], mode="lines",
#                     uncertainty=False, particle=False,
#                     track_label="Lévy RTS", line=dict(width=3, color=colors[2]))
# # plotter.fig.write_html(str(file_path))   # ⇐ uncomment to save
# plotter.fig.show()

In [179]:
# # 6.  Plotting velocity
# # ------------------------------------------------------------------
# file_path   = Path(folder_path) / "eurchfvplot.html"
# file_path.parent.mkdir(parents=True, exist_ok=True)

# velplotter = Plotterly(autosize=False, width=1500, height=800,
#                     dimension=Dimension.ONE, axis_labels=["Price"])

# # ---- aesthetics --------------------------------------------------
# velplotter.fig.update_layout(
#     plot_bgcolor="white",
#     xaxis=dict(showgrid=True, gridcolor="lightgray",
#                title=dict(text="Time", font=dict(size=20))),
#     yaxis=dict(showgrid=True, gridcolor="lightgray",
#                title=dict(text="dPrice/dt", font=dict(size=20))),
#     legend=dict(font=dict(size=15),
#                 bordercolor="Black", borderwidth=2,
#                 orientation='h')
# )
# # filtered tracks
# velplotter.plot_tracks(LP_track, [1], mode="lines",
#                     uncertainty=False, particle=False,
#                     track_label="Lévy filtered", line=dict(width=2))
# velplotter.plot_tracks(GP_track, [1], mode="lines",
#                     uncertainty=False, particle=False,
#                     track_label="Gaussian filtered", line=dict(width=2))
# # velplotter.plot_tracks(RTS_track, [1], mode="lines",
# #                     uncertainty=False, particle=False,
# #                     track_label="Levy RTS", line=dict(width=2))
# # plotter.fig.write_html(str(file_path))   # ⇐ uncomment to save
# velplotter.fig.show()